In [ ]:
# default_exp products

In [ ]:
#hide
# !pip install -q nbdev lambdasdk s3bz

In [ ]:
#hide
USER=None
PW=None

# Products

> interact with product apis

In [ ]:
#hide
from nbdev.showdoc import *
import logging

In [ ]:
#export
from s3bz.s3bz import S3
from botocore.config import Config
from lambdasdk.lambdasdk import Lambda, InvocationType
# one Event/Response source (vps-jg2): awsSchema's, which has Event.getInput; the local schema module is not imported here
from awsSchema.apigateway import Event, Response
from typing import Optional, List, Any
import ujson as json
from nicHelper.wrappers import add_method
from dataclasses_json import dataclass_json
from dataclasses import dataclass, fields
import logging, uuid, zlib, requests, boto3

log = logging.getLogger(__name__)

In [ ]:
#hide
logging.basicConfig(level=logging.INFO)

In [ ]:
#export
INTCOLS = ['iprcode','cprcode', 'oprcode', 'sellingPrice']
# vps-xh2: barcodes are identifiers, not numbers. int() would turn '0012345678905' (a UPC-A written as an EAN-13)
# into 12345678905, and the lost zero cannot be restored, so they are sent as strings of digits, unchanged.
BARCODECOLS = ['pr_barcode', 'pr_barcode2']



In [ ]:
#export
# validateConfig and FunctionNames are defined once, in names.ipynb (villaProductSdk/names.py) (vps-u5v)
from villaProductSdk.names import validateConfig, FunctionNames
_all_ = ['validateConfig', 'FunctionNames']

In [ ]:
#export
class ProductSdkError(Exception):
  '''raised when a product function call fails (vps-jg2): the invoke itself failed, the response could not be
  parsed, or the function answered with a status other than 200. Earlier versions printed and returned None or
  the error body as if it were data.'''
  def __init__(self, message:str, statusCode:Optional[int] = None, functionName:Optional[str] = None, body:Any = None):
    super().__init__(message)
    self.statusCode = statusCode
    self.functionName = functionName
    self.body = body
  def __str__(self):
    return f'{self.args[0]} (function={self.functionName}, statusCode={self.statusCode}, body={self.body!r})'


def parseLambdaResponse(lambdaResponse:Any, functionName:Optional[str] = None)->Any:
  '''the parsed body of an API Gateway style lambda response; ProductSdkError if it cannot be parsed or is not 200.
  The headers may be under 'headers' (API Gateway) or 'header' (the deployed product functions); lambdaResponse
  itself is left unchanged.'''
  if not isinstance(lambdaResponse, dict) or 'body' not in lambdaResponse:
    raise ProductSdkError('the lambda response has no body', functionName = functionName, body = lambdaResponse)
  # vps-8z7: Response.fromDict passes every key to the constructor, so the deployed functions' 'header', or a proxy
  # key Response has no field for (multiValueHeaders, isBase64Encoded), was a TypeError and every real answer a parse
  # error. fromDict gets a new dict of Response's own fields (body, statusCode and headers in every awsSchema release
  # settings.ini allows), and pops 'body' from that, not from the caller's dict. The fields are read here, not at
  # import, so the module still imports against stub packages (tests/test_function_names_single_source.py).
  responseFields = {f.name for f in fields(Response)}
  responseInput = {k: v for k, v in lambdaResponse.items() if k in responseFields}
  if 'header' in lambdaResponse and 'headers' in responseFields:
    responseInput.setdefault('headers', lambdaResponse['header'])
  try:
    response = Response.fromDict(responseInput)
  except Exception as e:
    raise ProductSdkError(f'the lambda response could not be parsed: {type(e).__name__}',
                          statusCode = lambdaResponse.get('statusCode'), functionName = functionName,
                          body = lambdaResponse.get('body')) from e
  if response.statusCode != 200:
    raise ProductSdkError('the lambda answered with an error', statusCode = response.statusCode,
                          functionName = functionName, body = response.body)
  return response.body

In [ ]:
#export
class ProductSdk:
  '''
    the main class for interacting with product endpoint
    user/pw are optional
  '''
  def __init__(self, 
               branch:str = 'dev-manual', 
               user:Optional[str] = None, 
               pw:Optional[str] = None,
               region:str = 'ap-southeast-1',
               connect_timeout:float = 10,
               read_timeout:float = 30):
    validateConfig(branch, region)
    self.branchName = branch
    self.functionNames = FunctionNames(branchName = branch)
    self.lambdaClient = Lambda(user =user, pw=pw, region = region)
    # vps-up7: lambdasdk builds its boto3 client with no Config, so a stalled call could hang forever; the same
    # client with timeouts and standard retries, under lambdasdk's own invoke()
    self.lambdaClient.lambdaClient = boto3.client(
      'lambda', region_name = region, aws_access_key_id = user, aws_secret_access_key = pw,
      config = Config(connect_timeout = connect_timeout, read_timeout = read_timeout,
                      retries = {'mode': 'standard', 'max_attempts': 3}))
    self.timeouts = (connect_timeout, read_timeout)
    self.user = user; self.pw = pw; self.region = region
    
  @staticmethod
  def returnLambdaResponse(lambdaResponse:dict, functionName:Optional[str] = None):
    '''the parsed body; raises ProductSdkError (it used to log and return None)'''
    return parseLambdaResponse(lambdaResponse, functionName)
      
  @staticmethod
  def printFirst(inputDict:dict):
    return next(iter(inputDict.items()))

  def _invoke(self, functionName:str, payload:Any, invocationType:str = InvocationType.requestResponse):
    '''invoke a function; a failed invoke (network, permissions, missing function) is a ProductSdkError'''
    try:
      return self.lambdaClient.invoke(functionName = functionName, input = payload, invocationType = invocationType)
    except Exception as e:
      raise ProductSdkError(f'invoking the lambda failed: {type(e).__name__}: {e}', functionName = functionName) from e
  
  def generalInvoke(self, functionName, payload):
    '''invoke and return the parsed body; ProductSdkError on a failed invoke, an unparseable or a non-200 response'''
    return parseLambdaResponse(self._invoke(functionName, payload), functionName)
    
  def updateWithS3(self, data, 
                   inputKeyName:Optional[str] = None, 
                   invocationType = InvocationType.event,
                   user= None, pw= None)->dict:
    '''upload products to the input bucket and start the ingestion function.

    Each call uploads under its own key (input-data-name-<uuid>) unless inputKeyName is given, so two callers
    never overwrite each other's upload (the fixed default key did). Returns
    {'key', 'bucket', 'functionName', 'invocationType', 'result'}: result is the parsed body for
    RequestResponse and None for an Event (async) invocation.'''
    _checkProducts(data)  # vps-28b: before anything is saved or invoked
    # put users if not specified
    user = user or self.user; pw = pw or self.pw
    key = inputKeyName or f'input-data-name-{uuid.uuid4().hex}'
    
    # extract function name and inputbucket name
    inputBucketName = self.functionNames.inputBucket()
    functionName = self.functionNames.updateS3()
    log.info(f'bucket is {inputBucketName}, key is {key}')
    
    # save data to s3
    ## convert keys to int
    filteredProducts = [{k:int(v) if k in INTCOLS else str(v) for k,v in product.items()}for product in data]
    
    try:
      S3.save(key = key, 
              objectToSave = filteredProducts , 
              bucket = inputBucketName,
              user=user, pw=pw)
    except Exception as e:
      raise ProductSdkError(f'saving the input to s3 failed: {type(e).__name__}: {e}', functionName = functionName) from e
    log.info('data is saved to s3, invoking ingestion function')
    
    # call lambda function
    inputValue = Event(body = json.dumps({ 'key': key })).to_dict()
    lambdaResponse = self._invoke(functionName, inputValue, invocationType)
    result = None if invocationType == InvocationType.event else parseLambdaResponse(lambdaResponse, functionName)
    return {'key': key, 'bucket': inputBucketName, 'functionName': functionName,
            'invocationType': invocationType, 'result': result}

  
  def allQuery(self):
    functionName = self.functionNames.allQuery()
    url = self.generalInvoke(functionName, {})['url']
    result = retrieveJsonZl(url, timeout = self.timeouts)  # vps-up7
    return result
  def syncS3(self):
    '''force s3 to sync with the newly input data'''
    return self.generalInvoke(self.functionNames.dumpToS3(), {})

In [ ]:
#export
def retrieveJsonZl(url, timeout = (10, 30)):
  '''download the zlib-compressed product table and read it with pandas (needed for allQuery only)'''
  response:requests.Response = requests.get(url, timeout = timeout)  # vps-up7
  if response.status_code != 200:
    raise ProductSdkError('downloading the product table failed', statusCode = response.status_code, body = response.text[:500])
  import io
  import pandas as pd
  jsonDb = zlib.decompress(response.content).decode()
  # vps-0qr: pandas 2.1 deprecated passing literal JSON as a str and pandas 3 reads a str as a file path
  return pd.read_json(io.StringIO(jsonDb), orient='split')

# Testing

In [ ]:
from dataclasses import dataclass
from dataclasses_json import dataclass_json
from random import randrange
from datetime import datetime
from pprint import pprint
import boto3
import pandas as pd


## generate dummy data for testing

In [ ]:
#Dummy Data
sampleProducts = [{'cprcode': '0171670', 'iprcode': '0171670', 'oprcode': '0171670', 'ordertype': 'Y', 'pr_abb': 'JIRAPAT YOUG KALE 2', 'pr_active': 'Y', 'pr_cgcode': '05', 'pr_code': '0171670', 'pr_dpcode': '19', 'pr_engname': 'JIRAPAT YOUNG KALE 200 G.', 'pr_ggcode': '057', 'pr_market': 'JIRAPAT ยอดคะน้า 200 G.', 'pr_name': 'JIRAPAT ยอดคะน้า 200 G.', 'pr_puqty': '1', 'pr_sa_method': '1', 'pr_sucode1': 'CM845     ', 'pr_suref3': 'A', 'prtype': 'I', 'psqty': '1', 'pstype': '1'}, {'cprcode': '0235141', 'iprcode': '0235141', 'oprcode': '0235141', 'ordertype': 'Y', 'pr_abb': 'EEBOO-PZCT3-PUZZLE', 'pr_active': 'Y', 'pr_cgcode': '08', 'pr_code': '0235141', 'pr_dpcode': '19', 'pr_engname': 'EEBOO,ANIMAL COUNTING PUZZLE_3ED,PZCT3', 'pr_ggcode': '113', 'pr_market': 'eeboo,PUZZLE-PZCT3', 'pr_name': 'EEBOO-PZCT3-ตัวต่อนับเลข ANIMAL COUNTING_3ED', 'pr_puqty': '1', 'pr_sa_method': '1', 'pr_sucode1': 'CM1979    ', 'pr_suref3': 'A', 'prtype': 'I', 'psqty': '1', 'pstype': '1'}, {'cprcode': '0217153', 'iprcode': '0217153', 'oprcode': '0217153', 'ordertype': 'Y', 'pr_abb': 'COCOA LOCO MILK CHOC', 'pr_active': 'Y', 'pr_cgcode': '98', 'pr_code': '0217153', 'pr_dpcode': '28', 'pr_engname': 'COCOA LOCO MILK CHOCOLATE OWL LOLLY 26G.', 'pr_ggcode': '003', 'pr_market': 'COCOA LOCO MILK CHOCOLATE OWL', 'pr_name': 'COCOA LOCO MILK CHOCOLATE OWL LOLLY 26G.', 'pr_puqty': '24', 'pr_sa_method': '1', 'pr_sucode1': 'F1222     ', 'pr_suref3': 'S', 'prtype': 'I', 'psqty': '1', 'pstype': '1'}, {'cprcode': '0182223', 'iprcode': '0182223', 'oprcode': '0182223', 'ordertype': 'Y', 'pr_abb': 'CIRIO PIZZASSIMO 400', 'pr_active': 'Y', 'pr_cgcode': '06', 'pr_code': '0182223', 'pr_dpcode': '06', 'pr_engname': 'CIRIO PIZZASSIMO 400G.', 'pr_ggcode': '004', 'pr_market': 'CIRIO ซอสทำพิซซ่า 400 G.', 'pr_name': 'CIRIO ซอสทำพิซซ่า 400 G.', 'pr_puqty': '12', 'pr_sa_method': '1', 'pr_sucode1': '2589      ', 'pr_suref3': 'C', 'prtype': 'I', 'psqty': '1', 'pstype': '1'}, {'cprcode': '0124461', 'iprcode': '0124461', 'oprcode': '0124461', 'ordertype': 'Y', 'pr_abb': 'NEW CHOICE LYCHEE', 'pr_active': 'Y', 'pr_cgcode': '02', 'pr_code': '0124461', 'pr_dpcode': '02', 'pr_engname': 'NEW CHOICE LYCHEE', 'pr_ggcode': '003', 'pr_market': 'NEW CHOICE กลิ่นลิ้นจี่', 'pr_name': 'NEW CHOICE กลิ่นลิ้นจี่', 'pr_puqty': '12', 'pr_sa_method': '1', 'pr_sucode1': '695       ', 'pr_suref3': 'A', 'prtype': 'I', 'psqty': '1', 'pstype': '1'}]
# sampleProducts = [{k: int(v) if k in INTCOLS else v for k,v in product.items()}for product in sampleProducts]

## Create main class object

In [ ]:
sdk = ProductSdk(branch = 'dev-manual')

## Upload s3 data

In [ ]:
#hide

In [ ]:
%%time
result = sdk.updateWithS3(
    sampleProducts,
    invocationType = InvocationType.requestResponse
  )
print(result['key'], result['result'])

### test with lots of data

In [ ]:
df = pd.read_csv('./tests/fixtures/products_sample.csv').fillna('none').astype('string')
for col in df.columns:
  if col in INTCOLS:
    df[col] = df[col].astype(int)
data = json.loads(df.to_json(orient='records'))
data[:1]

# data = list(pd.read_csv('./tests/fixtures/products_sample.csv').fillna('none').astype('string').T.to_dict().values())

/home/ec2-user/SageMaker/.persisted_conda/python38/lib/python3.8/site-packages/IPython/core/interactiveshell.py:3146: DtypeWarning: Columns (6,7,10,20,31) have mixed types.Specify dtype option on import or set low_memory=False.
  has_raised = await self.run_ast_nodes(code_ast.body, cell_name,


[{'cprcode': 9,
  'iprcode': 9,
  'oprcode': 9,
  'ordertype': 'Y',
  'pr_abb': 'J. WALKER RED 70 CL.',
  'pr_active': 'Y',
  'pr_cgcode': '1',
  'pr_code': '9',
  'pr_dpcode': '1.0',
  'pr_engname': 'JOHNNIE WALKER RED 70 CL.',
  'pr_ggcode': '1',
  'pr_market': 'J. WALKER RED 70 CL',
  'pr_name': 'JOHNNIE WALKER RED 70 CL',
  'pr_puqty': '12.0',
  'pr_sa_method': '1',
  'pr_sucode1': '1782',
  'pr_suref3': 'A',
  'prtype': 'I',
  'psqty': '1.0',
  'pstype': '1.0',
  'pr_country_th': 'none',
  'pr_country_en': 'United Kingdom',
  'pr_keyword_th': 'none',
  'pr_keyword_en': 'none',
  'pr_filter_th': 'Alcohol,Spirits  Liqueurs',
  'pr_filter_en': 'Alcohol,Spirits  Liqueurs',
  'online_category_l1_th': 'เบียร์, ไวน์และสุรา',
  'online_category_l1_en': 'Beer Wine & Spirits',
  'online_category_l2_th': 'เหล้า',
  'online_category_l2_en': 'Spirits & Liqueurs',
  'online_category_l3_th': 'none',
  'online_category_l3_en': 'none',
  'villa_category_l1_en': 'Dry Grocery',
  'villa_category_l2_

In [ ]:
%%time
result = sdk.updateWithS3(
    data,
    invocationType = InvocationType.event
  )
result

INFO:root:bucket is input-product-bucket-dev-manual
INFO:root:using accelerate endpoint
INFO:root:data was saved to s3
INFO:root:data is saved to s3, invoking ingestion function
INFO:root:input to lambda is {'body': '{"key":"input-data-name"}', 'headers': {}, 'statusCode': 200}
INFO:root:lambdaResponse is True


CPU times: user 17.3 s, sys: 272 ms, total: 17.5 s
Wall time: 17.8 s


'successfully sent event, please watch your slack'

In [ ]:
sdk.allQuery()

,cprcode,iprcode,oprcode,ordertype,pr_abb,pr_active,pr_cgcode,pr_code,pr_dpcode,pr_engname,...,pr_keyword_th,pr_name_en,pr_name_th,pr_online_name_en,pr_online_name_th,sort_weight,villa_category_l1_en,villa_category_l2_en,villa_category_l3_en,villa_category_l4_en
0,182223,182223,182223,Y,CIRIO PIZZASSIMO 400,Y,6,182223,6,CIRIO PIZZASSIMO 400G.,...,None,None,None,None,None,NaN,None,None,None,None
1,9,9,9,Y,J. WALKER RED 70 CL.,Y,1,9,1,JOHNNIE WALKER RED 70 CL.,...,none,JOHNNIE WALKER RED 70 CL.,JOHNNIE WALKER RED 70 CL,Johnnie Walker Red 700ml,JOHNNIE WALKER RED 700ml,0.0,Dry Grocery,Alcohol & Beverage,Whisky,Scoth Whisky
2,235141,235141,235141,Y,EEBOO-PZCT3-PUZZLE,Y,8,235141,19,"EEBOO,ANIMAL COUNTING PUZZLE_3ED,PZCT3",...,None,None,None,None,None,NaN,None,None,None,None
3,217153,217153,217153,Y,COCOA LOCO MILK CHOC,Y,98,217153,28,COCOA LOCO MILK CHOCOLATE OWL LOLLY 26G.,...,None,None,None,None,None,NaN,None,None,None,None
4,28,28,28,Y,J B RARE WHISKY 750,Y,1,28,1,J B RARE WHISKY 750 ML.,...,none,J B RARE WHISKY 750 ML.,J B RARE WHISKY 750 ML,J&B Rare Whisky 750ml,J&B Rare Whisky 750ml,0.0,Dry Grocery,Alcohol & Beverage,Whisky,Scoth Whisky
5,12,12,12,Y,J.WALER BLACK 70 CL,Y,1,12,1,JOHNNIE WALKER BLACK 70 CL.,...,none,JOHNNIE WALKER BLACK 70 CL.,JOHNNIE WALKER BLACK 70 CL.,Johnnie Walker Black 70 Cl.,JOHNNIE WALKER BLACK 70 CL.,0.0,Dry Grocery,Alcohol & Beverage,Whisky,none
6,33,33,33,Y,BALLANTINE S 700ml.,Y,1,33,1,BALLANTINE S FINEST 700 ML.,...,none,BALLANTINE S FINEST 700 ML.,BALLANTINE S FINEST 700 ML,Ballentine's Finest Whisky 700ml,Ballentine's Finest Whisky 700ml,0.0,Dry Grocery,Alcohol & Beverage,Whisky,none
7,26,26,26,Y,JOHN JAMESON 70 CL,Y,1,26,1,JOHN JAMESON 70 CL WHISKY,...,none,JOHN JAMESON 70 CL WHISKY,JOHN JAMESON 70 CL WHISKY,John Jameson Whisky 700ml,John Jameson Whisky 700ml,0.0,Dry Grocery,Alcohol & Beverage,Whisky,none
8,124461,124461,124461,Y,NEW CHOICE LYCHEE,Y,2,124461,2,NEW CHOICE LYCHEE,...,None,None,None,None,None,NaN,None,None,None,None
9,171670,171670,171670,Y,JIRAPAT YOUG KALE 2,Y,5,171670,19,JIRAPAT YOUNG KALE 200 G.,...,None,None,None,None,None,NaN,None,None,None,None


## Query Single Product

In [ ]:
#export
@add_method(ProductSdk)
def querySingleProduct(self, iprcode = '0171670', user=None, pw=None):
  '''query a single product; ProductSdkError when the call fails (it used to print and return None)'''
  _checkIprcode(iprcode)  # vps-28b
  return self.generalInvoke(self.functionNames.singleQuery(), Event.getInput({'iprcode': iprcode}))

In [ ]:
result = sdk.querySingleProduct('0171670')
result

{'cprcode': '0171670',
 'iprcode': '0171670',
 'oprcode': '0171670',
 'ordertype': 'Y',
 'pr_abb': 'JIRAPAT YOUNG KALE 2',
 'pr_active': 'Y',
 'pr_cgcode': '05',
 'pr_code': '0171670',
 'pr_dpcode': '19',
 'pr_engname': 'JIRAAT YOUNG KALE 200 G.',
 'pr_ggcode': '057',
 'pr_market': 'JIRAPAT ยอดคะน้า 200 G.',
 'pr_name': 'JIRAPAT ยอดคะน้า 200 G.',
 'pr_puqty': '1',
 'pr_sa_method': '1',
 'pr_sucode1': 'CM845',
 'pr_suref3': 'A',
 'prtype': 'I',
 'psqty': '1',
 'pstype': '1',
 'pr_country_th': 'none',
 'pr_country_en': 'none',
 'pr_keyword_th': 'none',
 'pr_keyword_en': 'none',
 'pr_filter_th': 'none',
 'pr_filter_en': 'none',
 'online_category_l1_th': 'none',
 'online_category_l1_en': 'none',
 'online_category_l2_th': 'none',
 'online_category_l2_en': 'none',
 'online_category_l3_th': 'none',
 'online_category_l3_en': 'none',
 'villa_category_l1_en': 'none',
 'villa_category_l2_en': 'none',
 'villa_category_l3_en': 'none',
 'villa_category_l4_en': 'none',
 'content_en': '0171670 JIRAPAT

In [ ]:
## failure query product notn found
result = sdk.querySingleProduct('3203290')
result

{'error': 'product not found'}

## All Query

In [ ]:
result = sdk.allQuery()
print(result.shape)
result.head()

(711, 50)


,cprcode,iprcode,oprcode,ordertype,pr_abb,pr_active,pr_cgcode,pr_code,pr_dpcode,pr_engname,...,pr_keyword_th,pr_name_en,pr_name_th,pr_online_name_en,pr_online_name_th,sort_weight,villa_category_l1_en,villa_category_l2_en,villa_category_l3_en,villa_category_l4_en
0,182223,182223,182223,Y,CIRIO PIZZASSIMO 400,Y,6,182223,6,CIRIO PIZZASSIMO 400G.,...,None,None,None,None,None,NaN,None,None,None,None
1,9,9,9,Y,J. WALKER RED 70 CL.,Y,1,9,1,JOHNNIE WALKER RED 70 CL.,...,none,JOHNNIE WALKER RED 70 CL.,JOHNNIE WALKER RED 70 CL,Johnnie Walker Red 700ml,JOHNNIE WALKER RED 700ml,0.0,Dry Grocery,Alcohol & Beverage,Whisky,Scoth Whisky
2,235141,235141,235141,Y,EEBOO-PZCT3-PUZZLE,Y,8,235141,19,"EEBOO,ANIMAL COUNTING PUZZLE_3ED,PZCT3",...,None,None,None,None,None,NaN,None,None,None,None
3,217153,217153,217153,Y,COCOA LOCO MILK CHOC,Y,98,217153,28,COCOA LOCO MILK CHOCOLATE OWL LOLLY 26G.,...,None,None,None,None,None,NaN,None,None,None,None
4,28,28,28,Y,J B RARE WHISKY 750,Y,1,28,1,J B RARE WHISKY 750 ML.,...,none,J B RARE WHISKY 750 ML.,J B RARE WHISKY 750 ML,J&B Rare Whisky 750ml,J&B Rare Whisky 750ml,0.0,Dry Grocery,Alcohol & Beverage,Whisky,Scoth Whisky


## Query list of items

In [ ]:
#export
@dataclass_json
@dataclass
class ProductsFromList:
  iprcodes: List[str]

In [ ]:
#export
@add_method(ProductSdk)
def queryList(self,iprcodes:List[str])->List[dict]:
  '''query several products; ProductSdkError when the call fails'''
  _checkIprcodes(iprcodes)  # vps-28b
  return self.generalInvoke(functionName=self.functionNames.listQuery() ,payload=Event.getInput({'iprcodes': iprcodes}))

In [ ]:
cprcodes = ['0217153','203915','0000009']
sdk.queryList(cprcodes)

{'errorMessage': "'cprcode'", 'errorType': 'KeyError', 'stackTrace': ['  File "/opt/python/villaProductDatabase/database.py", line 104, in lambdaProductsFromList\n    result:pd.DataFrame = ProductDatabase.productsFromList(productsFromList.iprcodes)\n', '  File "/opt/python/nicHelper/wrappers.py", line 37, in wrapper\n    return func( cls, *args, **kwargs)\n', '  File "/opt/python/villaProductDatabase/query.py", line 66, in productsFromList\n    products = db[db[\'cprcode\'].isin(cprcodes)]\n', '  File "/opt/python/pandas/core/frame.py", line 2906, in __getitem__\n    indexer = self.columns.get_loc(key)\n', '  File "/opt/python/pandas/core/indexes/base.py", line 2900, in get_loc\n    raise KeyError(key) from err\n']}


In [ ]:
json.dumps(Event.getInput({'iprcodes':iprcodes}))

'{"body":"{\\"iprcodes\\":[\\"0217153\\",\\"203915\\",\\"0000009\\"]}","headers":{},"statusCode":200}'

## Trigger s3 sync

In [ ]:
%%time
response = sdk.syncS3()
response

CPU times: user 15 ms, sys: 79 µs, total: 15 ms
Wall time: 7.6 s


{'result': {'cprcode': 182223,
  'iprcode': 182223,
  'oprcode': 182223,
  'ordertype': 'Y',
  'pr_abb': 'CIRIO PIZZASSIMO 400',
  'pr_active': 'Y',
  'pr_cgcode': '06',
  'pr_code': '0182223',
  'pr_dpcode': '06',
  'pr_engname': 'CIRIO PIZZASSIMO 400G.',
  'pr_ggcode': '004',
  'pr_market': 'CIRIO ซอสทำพิซซ่า 400 G.',
  'pr_name': 'CIRIO ซอสทำพิซซ่า 400 G.',
  'pr_puqty': '12',
  'pr_sa_method': '1',
  'pr_sucode1': '2589',
  'pr_suref3': 'C',
  'prtype': 'I',
  'psqty': '1',
  'pstype': '1',
  'content_en': None,
  'content_th': None,
  'hema_brand_en': None,
  'hema_brand_th': None,
  'hema_name_en': None,
  'hema_name_th': None,
  'hema_sizedesc': None,
  'online_category_l1_en': None,
  'online_category_l1_th': None,
  'online_category_l2_en': None,
  'online_category_l2_th': None,
  'online_category_l3_en': None,
  'online_category_l3_th': None,
  'pr_brand_en': None,
  'pr_brand_th': None,
  'pr_country_en': None,
  'pr_country_th': None,
  'pr_filter_en': None,
  'pr_filter_th

In [ ]:
#export
import re as _re


def _checkIprcode(code, name:str = 'iprcode'):
  '''vps-28b: a product code is a non-empty str of digits 0-9 or a non-negative int (never a bool)'''
  if isinstance(code, bool) or not isinstance(code, (str, int)):
    raise TypeError(f'{name} must be a str of digits or an int, not {type(code).__name__}')
  if isinstance(code, int):
    if code < 0:
      raise ValueError(f'{name} must not be negative')
  elif not _re.fullmatch(r'[0-9]+', code):
    raise ValueError(f'{name} must be a non-empty string of digits 0-9')


def _checkIprcodes(iprcodes):
  '''a non-empty list or tuple of product codes'''
  if not isinstance(iprcodes, (list, tuple)):
    raise TypeError(f'iprcodes must be a list or tuple of codes, not {type(iprcodes).__name__}')
  if not iprcodes:
    raise ValueError('iprcodes must not be empty')
  for i, code in enumerate(iprcodes):
    _checkIprcode(code, f'iprcodes[{i}]')


def _checkProducts(data):
  '''a non-empty list of product dicts, each with a valid iprcode and, when present, valid barcodes (vps-xh2)'''
  if not isinstance(data, list):
    raise TypeError(f'data must be a list of product dicts, not {type(data).__name__}')
  if not data:
    raise ValueError('data must not be empty')
  for i, product in enumerate(data):
    if not isinstance(product, dict):
      raise TypeError(f'data[{i}] must be a dict, not {type(product).__name__}')
    if 'iprcode' not in product:
      raise ValueError(f'data[{i}] has no iprcode')
    _checkIprcode(product['iprcode'], f'data[{i}].iprcode')
    for col in BARCODECOLS:  # vps-xh2: a str of digits or an int, sent as that string; leave the key out for none
      if col in product:
        _checkIprcode(product[col], f'data[{i}].{col}')

In [ ]:
#hide
from nbdev.export import *
notebook2script()

In [ ]:
%time
import requests
from io import BytesIO
from PIL import Image

##large Image
url = 'http://d19oj5aeuefgv.cloudfront.net'
sku = '0189194'
Image.open(BytesIO(requests.get(f'{url}/{sku}').content),formats=['PNG'])

In [ ]:
%time
url = 'https://d1vl5j0v241n75.cloudfront.net'
sku = '0189194'
Image.open(BytesIO(requests.get(f'{url}/{sku}').content),formats=['PNG'])